# Customer Churn & CLV — Data Cleaning

This notebook cleans the raw transactional data and prepares two datasets for the rest of the project:

- **Sales:** valid purchases made by identified customers.
- **Returns:** cancellations, which I keep separately because they can be a signal of customer dissatisfaction.

In [1]:
# Import necessary libraries
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
# Load the dataset (both sheets of the Excel file)
DATA_PATH = Path("../data/raw/online_retail_II.xlsx")

sheets = pd.read_excel(DATA_PATH, sheet_name=None)
retail_df = pd.concat(sheets.values(), ignore_index=True)

retail_df.shape

(1067371, 8)

In [3]:
# Rename the columns to make them easier to use
retail_df = retail_df.rename(columns={"Customer ID": "CustomerID"})

# Make sure Invoice and StockCode are text (some codes are numbers and others are letters)
retail_df["Invoice"] = retail_df["Invoice"].astype(str)
retail_df["StockCode"] = retail_df["StockCode"].astype(str)

retail_df.dtypes

Invoice                   str
StockCode                 str
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
CustomerID            float64
Country                   str
dtype: object

## 1. Duplicates

In [4]:
# Drop the duplicated rows (mainly the overlap between the two sheets)
rows_before = len(retail_df)

retail_df = retail_df.drop_duplicates()

print(f"Rows removed: {rows_before - len(retail_df)}")
print("Number of duplicates:", retail_df.duplicated().sum())

Rows removed: 34335
Number of duplicates: 0


## 2. Missing Customer ID

The goal of the project is to predict which **customers** will stop buying. A purchase without a customer can't be followed over time, so these rows are removed.

In [5]:
# Remove the rows without CustomerID
rows_before = len(retail_df)

retail_df = retail_df[retail_df["CustomerID"].notna()].copy()
retail_df["CustomerID"] = retail_df["CustomerID"].astype(int)

print(f"Rows removed: {rows_before - len(retail_df)}")

Rows removed: 235151


## 3. Non-product stock codes

In [6]:
# Define the stock codes that are not products (postage, fees, adjustments, discounts...)
NON_PRODUCT = [
    "POST", "DOT", "M", "C2", "D", "S", "BANK CHARGES", "ADJUST", "ADJUST2",
    "AMAZONFEE", "CRUK", "PADS", "B", "TEST001", "TEST002"
]

rows_before = len(retail_df)

retail_df = retail_df[~retail_df["StockCode"].isin(NON_PRODUCT)]

print(f"Rows removed: {rows_before - len(retail_df)}")

Rows removed: 3660


## 4. Revenue and cancellations

In [7]:
# Calculate the revenue of each line
retail_df["Revenue"] = retail_df["Quantity"] * retail_df["Price"]

# Identify the cancellations
retail_df["is_cancellation"] = retail_df["Invoice"].str.startswith("C")

retail_df["is_cancellation"].value_counts()

is_cancellation
False    776639
True      17586
Name: count, dtype: int64

In [8]:
# Separate the returns (cancellations) into their own dataset
returns_df = retail_df[retail_df["is_cancellation"]].copy()

returns_df.shape

(17586, 10)

In [9]:
# Keep only valid sales: not cancelled, positive quantity and positive price
sales_df = retail_df[
    (~retail_df["is_cancellation"]) &
    (retail_df["Quantity"] > 0) &
    (retail_df["Price"] > 0)
].copy()

sales_df = sales_df.drop(columns=["is_cancellation"])
returns_df = returns_df.drop(columns=["is_cancellation"])

sales_df.shape

(776579, 9)

In [10]:
# Final check of the clean sales dataset
print(f"Sales lines: {len(sales_df)}")
print(f"Customers: {sales_df['CustomerID'].nunique()}")
print(f"Invoices: {sales_df['Invoice'].nunique()}")
print(f"Total revenue: £{sales_df['Revenue'].sum():,.0f}")
print(f"Missing values: {sales_df.isna().sum().sum()}")

Sales lines: 776579
Customers: 5852
Invoices: 36594
Total revenue: £17,068,583
Missing values: 0


In [11]:
# As a final step, save the clean datasets for the next notebooks
OUTPUT_SALES = Path("../data/processed/sales_clean.csv")
OUTPUT_RETURNS = Path("../data/processed/returns_clean.csv")

OUTPUT_SALES.parent.mkdir(parents=True, exist_ok=True)

sales_df.to_csv(OUTPUT_SALES, index=False)
returns_df.to_csv(OUTPUT_RETURNS, index=False)

print(f"Clean sales saved to: {OUTPUT_SALES} {sales_df.shape}")
print(f"Returns saved to: {OUTPUT_RETURNS} {returns_df.shape}")

Clean sales saved to: ../data/processed/sales_clean.csv (776579, 9)
Returns saved to: ../data/processed/returns_clean.csv (17586, 9)


## Conclusions

### Cleaning performed

- Exact duplicated rows were removed (mainly the overlap between the two Excel sheets).
- Rows without Customer ID were removed, because they can't be followed over time.
- Non-product stock codes (postage, fees, manual adjustments) were removed.
- Revenue was calculated as `Quantity × Price`.
- Cancellations were saved separately as returns, to be used later as a feature.
- Only valid sales (positive quantity and price) were kept.

### Result

- **776,579 sales lines** from **5,852 customers** and **36,594 invoices**, with a total revenue of about **£17.1M**.